# SmartEdu AI: Sequential Academic Risk Prediction Benchmark
### Comparing Rule-Based Heuristics, Classical Machine Learning, and Attention-Based Sequential LSTM

> **Dataset Transparency Notice:** The longitudinal weekly dataset used in this benchmark represents proxy / synthetic academic trajectories generated across 16 weeks to simulate real LMS engagement patterns. Real-world institutional deployment requires collecting and logging longitudinal data across active semesters.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure ml_pipeline is accessible
sys.path.insert(0, os.path.abspath('.'))
from generate_dataset import save_dataset, generate_student_cohort
from feature_engineering import (
    load_weekly_csv,
    extract_student_subject_sequences,
    extract_aggregated_static_features,
    evaluate_rule_based_baseline
)
from metrics import compute_classification_metrics, stratified_student_split, SimpleStandardScaler
from models import RuleBasedClassifier, train_logistic_regression, train_xgboost, StudentRiskLSTM, HAS_XGB
from explainability import explain_lstm_prediction

print(f"PyTorch Version: {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"XGBoost Available: {HAS_XGB}")

## 1. Longitudinal Sequence Generation & Feature Extraction
We extract 16-week sequences per student per subject with 5 normalized features:
1. `attendance_rate_this_week`
2. `cumulative_attendance_rate`
3. `assignment_submitted`
4. `cumulative_avg_score` (normalized)
5. `score_trend` (normalized delta vs prior week)

In [ ]:
data_dir = os.path.join(os.getcwd(), "data")
os.makedirs(data_dir, exist_ok=True)
weekly_csv = os.path.join(data_dir, "weekly_student_records.csv")

if not os.path.exists(weekly_csv):
    save_dataset(data_dir)

records = load_weekly_csv(weekly_csv)
print(f"Loaded {len(records)} weekly student records.")

# Extract sequential features for LSTM
X_seq, masks, seq_keys, y_seq, student_ids = extract_student_subject_sequences(records, max_timesteps=16)

# Extract static aggregated features for Classical ML
X_static, static_feature_names, static_keys, y_static, _ = extract_aggregated_static_features(records)

print(f"Sequential Matrix Shape: {X_seq.shape} (N, Timesteps=16, Features=5)")
print(f"Static Feature Vector Shape: {X_static.shape} (N, Features=7)")

## 2. Stratified Student-Level Group Split (Zero Data Leakage)
To prevent data leakage, all weeks and subjects belonging to the same student are strictly placed in either the Train partition OR the Test partition.

In [ ]:
train_mask, test_mask = stratified_student_split(student_ids, y_seq, test_ratio=0.20, random_seed=42)

X_seq_train, X_seq_test = X_seq[train_mask], X_seq[test_mask]
masks_train, masks_test = masks[train_mask], masks[test_mask]
y_train, y_test = y_seq[train_mask], y_seq[test_mask]
test_keys = [seq_keys[i] for i, flag in enumerate(test_mask) if flag]

scaler = SimpleStandardScaler()
X_stat_train_scaled = scaler.fit_transform(X_static[train_mask])
X_stat_test_scaled = scaler.transform(X_static[test_mask])

print(f"Train cohort: {len(y_train)} samples | Held-out test cohort: {len(y_test)} samples")

## 3. Training & Evaluating All Three Model Paradigms

In [ ]:
benchmark_results = {}

# Model A: Rule-Based Baseline
y_pred_rule = evaluate_rule_based_baseline(records, test_keys)
benchmark_results["Rule-Based Baseline"] = compute_classification_metrics(y_test, y_pred_rule)

# Model B1: Logistic Regression
lr_model = train_logistic_regression(X_stat_train_scaled, y_train, epochs=250, lr=0.015)
y_pred_lr = lr_model.predict(X_stat_test_scaled)
benchmark_results["Logistic Regression"] = compute_classification_metrics(y_test, y_pred_lr)

# Model B2: XGBoost
xgb_model = train_xgboost(X_stat_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_stat_test_scaled)
benchmark_results["XGBoost"] = compute_classification_metrics(y_test, y_pred_xgb)

# Model C: PyTorch Sequential LSTM with Attention
from train_and_evaluate import train_lstm_model
lstm_model = train_lstm_model(X_seq_train, y_train, masks_train, X_seq_test, y_test, masks_test, epochs=45, lr=0.001)

lstm_model.eval()
with torch.no_grad():
    t_x = torch.tensor(X_seq_test, dtype=torch.float32)
    t_m = torch.tensor(masks_test, dtype=torch.bool)
    logits, _ = lstm_model(t_x, mask=t_m)
    y_pred_lstm = torch.argmax(logits, dim=-1).cpu().numpy()

benchmark_results["Sequential LSTM (Ours)"] = compute_classification_metrics(y_test, y_pred_lstm)

## 4. Benchmark Performance Comparison

In [ ]:
print("=" * 75)
print(f"{'Model Architecture':<28} | {'Accuracy':<9} | {'Precision':<9} | {'Recall':<9} | {'Macro F1':<9}")
print("-" * 75)
for name, m in benchmark_results.items():
    print(f"{name:<28} | {m['Accuracy']:<9.4f} | {m['Macro-Precision']:<9.4f} | {m['Macro-Recall']:<9.4f} | {m['Macro-F1']:<9.4f}")
print("=" * 75)

## 5. Explainability & Temporal Attention Breakdown
Using learnable attention weights across the 16 semester weeks, we can visualize which specific weeks caused a student to be flagged as at-risk.

In [ ]:
sample_idx = 0
sample_x = torch.tensor(X_seq_test[sample_idx:sample_idx+1], dtype=torch.float32)
sample_m = torch.tensor(masks_test[sample_idx:sample_idx+1], dtype=torch.bool)

exp = explain_lstm_prediction(lstm_model, sample_x, mask_tensor=sample_m)
print(f"Predicted Risk Level: {['Low', 'Medium', 'High'][exp['predicted_class']]}")
print(f"Class Probabilities: {exp['class_probabilities']}")
print("\nTop Attended Weeks:")
for w in exp['top_weeks']:
    print(f"- Week {w['week']}: {w['attention_pct']}% attention weight")
print("\nTop Contributing Adverse Factors:")
for f in exp['top_factors']:
    print(f"- {f['label']} (Impact: {f['impact_score']})")